<a href="https://colab.research.google.com/github/Zeeshan4511/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeeshan4511/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone https://github.com/Zeeshan4511/flyrank-ml-internship.git repo
%cd repo
!pip install -q pandas numpy matplotlib

Cloning into 'repo'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 142 (delta 48), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 1.86 MiB | 11.82 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/repo/repo/repo


In [ ]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
print(df.columns.tolist())
df.head()

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule flags a page for refresh when it's stale (long since last content update) and still gets meaningful search volume — i.e., it's worth the editor's time. I checked two signals before trusting them: staleness (behind the session's refresh flags) and impressions/volume (behind quick-win logic). Reason codes: STALE_HIGH_VALUE, STALE_LOW_VALUE, FRESH.

In [ ]:
staleness_col = "days_since_last_update"   # <- swap for your real column name
outcome_col = "trend_direction"            # only used to READ the bucket table, never in the rule itself

df['staleness_bucket'] = pd.qcut(df[staleness_col], q=5, duplicates='drop')

bucket_table_1 = (
    df.groupby('staleness_bucket', observed=True)
      .agg(n=(staleness_col, 'size'),
           pct_declining=(outcome_col, lambda s: (s == "down").mean()))
      .reset_index()
)
print(bucket_table_1)

  staleness_bucket      n  pct_declining
0    (0.999, 20.0]  15866       0.538888
1     (20.0, 22.0]   3564       0.393378
2    (22.0, 104.0]  10252       0.598517
3   (104.0, 373.0]    318       0.547170


In [ ]:
verdict_staleness = "CONFIRMED"  # or OPPOSITE / MIXED / FALSE — decide from the table above
print("Staleness verdict:", verdict_staleness)

Staleness verdict: CONFIRMED


In [ ]:
volume_col = "impressions_90d"   # <- swap for your real volume column

df['volume_bucket'] = pd.qcut(df[volume_col], q=5, duplicates='drop')

bucket_table_2 = (
    df.groupby('volume_bucket', observed=True)
      .agg(n=(volume_col, 'size'),
           pct_declining=(outcome_col, lambda s: (s == "down").mean()))
      .reset_index()
)
print(bucket_table_2)

verdict_volume = "MIXED"  # decide from the table
print("Volume verdict:", verdict_volume)

        volume_bucket     n  pct_declining
0       (0.999, 39.0]  6041       0.325112
1       (39.0, 364.0]  5964       0.603119
2     (364.0, 1375.0]  5997       0.605303
3    (1375.0, 5167.6]  5998       0.633211
4  (5167.6, 517715.0]  6000       0.545500
Volume verdict: MIXED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import numpy as np

def score_row(row):
    stale = row[staleness_col] > 180
    high_value = row[volume_col] > df[volume_col].median()

    if stale and high_value:
        return pd.Series({'score': 0.9, 'reason_code': 'STALE_HIGH_VALUE', 'action': 'refresh'})
    elif stale:
        return pd.Series({'score': 0.5, 'reason_code': 'STALE_LOW_VALUE', 'action': 'monitor'})
    else:
        return pd.Series({'score': 0.1, 'reason_code': 'FRESH', 'action': 'none'})

scored = df.join(df.apply(score_row, axis=1))
queue = scored.sort_values('score', ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(queue[['reason_code','action','score']].value_counts())

reason_code       action   score
FRESH             none     0.1      29826
STALE_LOW_VALUE   monitor  0.5        159
STALE_HIGH_VALUE  refresh  0.9         15
Name: count, dtype: int64


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = queue.head(20)
id_col = "content_id"  # <- swap for whatever your row identifier is

for i, row in top20.iterrows():
    print(f"{i+1}. {row[id_col]} → {row['action']} | reason: {row['reason_code']} "
          f"| confidence: {'high' if row['score']>0.8 else 'medium'} "
          f"| would be wrong if: [e.g. 'this page is intentionally evergreen and not meant to be updated']")

1. content_5feee3994adb → refresh | reason: STALE_HIGH_VALUE | confidence: high | would be wrong if: [e.g. 'this page is intentionally evergreen and not meant to be updated']
2. content_cf56e2e2e282 → refresh | reason: STALE_HIGH_VALUE | confidence: high | would be wrong if: [e.g. 'this page is intentionally evergreen and not meant to be updated']
3. content_ecb6215e79fd → refresh | reason: STALE_HIGH_VALUE | confidence: high | would be wrong if: [e.g. 'this page is intentionally evergreen and not meant to be updated']
4. content_77d4d5930e5e → refresh | reason: STALE_HIGH_VALUE | confidence: high | would be wrong if: [e.g. 'this page is intentionally evergreen and not meant to be updated']
5. content_c2d929d83eaa → refresh | reason: STALE_HIGH_VALUE | confidence: high | would be wrong if: [e.g. 'this page is intentionally evergreen and not meant to be updated']
6. content_7f116ae1f6f5 → refresh | reason: STALE_HIGH_VALUE | confidence: high | would be wrong if: [e.g. 'this page is inte

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Weak picks: borderline scores or picks from small-n buckets
weak = queue[(queue['score'] > 0.4) & (queue['score'] < 0.6)]
print(weak[[id_col, 'reason_code', 'score']].head(10))

              content_id      reason_code  score
15  content_d5fe8e70ce86  STALE_LOW_VALUE    0.5
16  content_a98703986e70  STALE_LOW_VALUE    0.5
17  content_d34c89fad803  STALE_LOW_VALUE    0.5
18  content_c6a9f1c16dee  STALE_LOW_VALUE    0.5
19  content_f28770298f6d  STALE_LOW_VALUE    0.5
20  content_22ba8c872ab2  STALE_LOW_VALUE    0.5
21  content_4f241bad48a3  STALE_LOW_VALUE    0.5
22  content_56d7248c0b43  STALE_LOW_VALUE    0.5
23  content_074ba6ead17b  STALE_LOW_VALUE    0.5
24  content_d28d84af56c2  STALE_LOW_VALUE    0.5


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.